1. Load clean Datasets

In [7]:
import pandas as pd

# Load clean datasets
customers = pd.read_csv('../data/cleaned/customers_clean.csv')
offers = pd.read_csv('../data/cleaned/offers_clean.csv')
events = pd.read_csv('../data/cleaned/events_clean.csv')

2. Demographic Features - age_group, income group

In [8]:
age_bins = [17, 24, 34, 44, 54, 64, 200]
age_labels = [
    '18-24 Young Adults', '25-34 Early Career', '35-44 Young Families',
    '45-54 Mature Professionals', '55-64 Pre-Retirement', '65+ Retirees'
]
customers['age_group'] = pd.cut(customers['age'], bins=age_bins, labels=age_labels)

income_bins = [0, 50000, 75000, 100000, 200000]
income_labels = [
    '30-50k Lower-Middle', '50-75k Middle', '75-100k Upper-Middle', '100-120k Affluent'
]
customers['income_group'] = pd.cut(customers['income'], bins=income_bins, labels=income_labels)



3. Tenure feature — how long each customer has been a member

In [15]:
# became_member_on is currently an object type, we need to convert it to datetime type for further calculations
print(customers['became_member_on'].dtype)

customers['became_member_on'] = pd.to_datetime(customers['became_member_on'])


datetime64[ns]


In [13]:
print(customers['became_member_on'].dtype)

datetime64[ns]


In [14]:
# used the latest join date in the data as the reference point (not today's real date)
reference_date = customers['became_member_on'].max()
customers['tenure_days'] = (reference_date - customers['became_member_on']).dt.days

tenure_bins = [-1, 365, 1095, 1825, 10000]
tenure_labels = ['New (<1yr)', 'Established (1-3yr)', 'Loyal (3-5yr)', 'Veteran (5yr+)']
customers['tenure_group'] = pd.cut(customers['tenure_days'], bins=tenure_bins, labels=tenure_labels)



4. Offer funnel Matching 

 Goal: for every 'offer received' event, figure out whether it was:
    - viewed (within the offer's duration window)
    - completed AFTER being viewed (a truly "influenced" completion,
        not just a coincidental purchase that happened to clear the threshold)

        

In [16]:
events.head()

,customer_id,event,time,offer_id,amount,reward_value,time_days
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,0.0
1,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,0.0
2,389bc3fa690240e798340f5a15918d5c,offer received,0,f19421c1d4aa40978ebb69ca19b0e20d,NaN,NaN,0.0
3,2eeac8d8feae4a8cad5a6af0499a211d,offer received,0,3f207df678b143eea3cee63160fa8bed,NaN,NaN,0.0
4,aa4862eba776480b8bb9c68455b8c2e1,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,0.0
